<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/promptEngineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-genai pandas

In [ ]:
import os
import json
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata

# Initialize the official Gemini Client using the Colab Secret
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("🚀 Gemini 2.5 Flash Client successfully initialized!")
except Exception as e:
    print(f"Error initializing client. Please check your Colab Secrets configuration. Details: {e}")

🚀 Gemini 2.5 Flash Client successfully initialized!


In [ ]:
# Storage for our deliverables report data
experiment_results = []

def run_experiment(task_name, description, system_instruction, user_prompt, config):
    """Helper function to execute calls and save results for analysis"""
    print(f"🔄 Running: {task_name}...")
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=config.get('temperature', 0.5),
                max_output_tokens=config.get('max_output_tokens', 400),
                response_mime_type=config.get('response_mime_type', 'text/plain'),
                stop_sequences=config.get('stop_sequences', None)
            )
        )

        result = {
            "Task": task_name,
            "Description": description,
            "Prompt Strategy": f"System: {system_instruction[:80]}... | User: {user_prompt}",
            "Settings": str(config),
            "Bot Response": response.text.strip()
        }
        experiment_results.append(result)

        print(f"✅ Finished: {task_name}\n" + "-"*40)
    except Exception as e:
        print(f"❌ Error during {task_name}: {e}")

# ==========================================
# TEST CASE 1: Contextual Prompting
# ==========================================
# We test how adding explicit domain data (Syllabus/Duration) prevents hallucinations or vague replies.
test_query = "What happens if I miss the final project deadline?"

run_experiment(
    task_name="1. Contextual Prompting (No Context)",
    description="Basic zero-shot response without background course information.",
    system_instruction="You are a university course assistant chatbot.",
    user_prompt=test_query,
    config={"temperature": 0.2}
)

course_context = """
You are a helpful student assistant for 'CS-101: Intro to AI'.
Course Duration: 12 weeks.
Syllabus details: Weeks 1-4 cover basic Python, Weeks 5-8 cover basic Math, Weeks 9-12 cover Machine Learning basics.
Deadlines & Grading Policy: All project assignments must be submitted via the student portal. Missing the final project deadline results in an automatic 20% point deduction per day, up to a maximum of 2 days late, after which a grade of zero is assigned. No extensions are granted unless accompanied by a medical exemption.
"""

run_experiment(
    task_name="2. Contextual Prompting (With Context)",
    description="Testing response accuracy when injected with complete course rules.",
    system_instruction=course_context,
    user_prompt=test_query,
    config={"temperature": 0.2}
)

# ==========================================
# TEST CASE 2: Format Constraints
# ==========================================
# Enforcing structured output formats (JSON vs Markdown bullet points)
format_query = "List the weekly breakdown of the course syllabus."

run_experiment(
    task_name="3. Format Constraints (Markdown Structure)",
    description="Forcing clean markdown formatting with strict bullet points.",
    system_instruction=course_context + "\nConstraint: You must only respond using structured Markdown formatting, utilizing clear bold headings and bulleted lists. Do not write standard paragraphs.",
    user_prompt=format_query,
    config={"temperature": 0.2}
)

run_experiment(
    task_name="4. Format Constraints (Strict JSON Output)",
    description="Forcing valid JSON structured output using Gemini's response_mime_type parameter.",
    system_instruction=course_context + "\nConstraint: You are a backend API data provider. You must return information strictly structured as valid JSON. Use keys: 'course_name', 'week_range', and 'topics_covered'. Do not wrap the JSON output in markdown code blocks.",
    user_prompt=format_query,
    config={"temperature": 0.1, "response_mime_type": "application/json"}
)

# ==========================================
# TEST CASE 3: Iterative Refinement
# ==========================================
# Refining the interaction by asking for specific action items and structural additions.
refinement_query = "How do I get help if I am stuck on a Python coding assignment?"

run_experiment(
    task_name="5. Iterative Refinement (Basic Answer)",
    description="Standard helpful baseline answer.",
    system_instruction=course_context,
    user_prompt=refinement_query,
    config={"temperature": 0.5}
)

refined_context = course_context + """
When students ask for help with coding:
1. Provide an encouraging environment.
2. Directly provide the official Discord link (discord.gg/cs101-ai) and TA office hours (Mon/Wed 2-4 PM).
3. Give them a 3-step blueprint to debug their problem before posting: (Step 1: Read the Traceback error, Step 2: Use print statements, Step 3: Check documentation).
"""

run_experiment(
    task_name="6. Iterative Refinement (Enriched with Resources)",
    description="Refined instructions containing resource URLs and specific multi-step debugging frameworks.",
    system_instruction=refined_context,
    user_prompt=refinement_query,
    config={"temperature": 0.5}
)

# ==========================================
# TEST CASE 4: Parameter Control (Temperature Exploration)
# ==========================================
# Direct analysis of low vs high temperature settings using an open-ended question.
creative_query = "Can you give me an engaging, real-world analogy to explain why we study AI?"

run_experiment(
    task_name="7. Parameter Control (Low Temp = 0.2)",
    description="Deterministic, focused, and predictable language optimization.",
    system_instruction=course_context,
    user_prompt=creative_query,
    config={"temperature": 0.2}
)

run_experiment(
    task_name="8. Parameter Control (High Temp = 0.9)",
    description="Highly creative, diverse vocabulary, and varied conceptual analogy.",
    system_instruction=course_context,
    user_prompt=creative_query,
    config={"temperature": 0.9}
)

print("🎉 All experiments executed successfully!")

🔄 Running: 1. Contextual Prompting (No Context)...
✅ Finished: 1. Contextual Prompting (No Context)
----------------------------------------
🔄 Running: 2. Contextual Prompting (With Context)...
✅ Finished: 2. Contextual Prompting (With Context)
----------------------------------------
🔄 Running: 3. Format Constraints (Markdown Structure)...
✅ Finished: 3. Format Constraints (Markdown Structure)
----------------------------------------
🔄 Running: 4. Format Constraints (Strict JSON Output)...
✅ Finished: 4. Format Constraints (Strict JSON Output)
----------------------------------------
🔄 Running: 5. Iterative Refinement (Basic Answer)...
✅ Finished: 5. Iterative Refinement (Basic Answer)
----------------------------------------
🔄 Running: 6. Iterative Refinement (Enriched with Resources)...
✅ Finished: 6. Iterative Refinement (Enriched with Resources)
----------------------------------------
🔄 Running: 7. Parameter Control (Low Temp = 0.2)...
❌ Error during 7. Parameter Control (Low Tem

In [ ]:
# Convert findings list into a DataFrame for presentation
df = pd.DataFrame(experiment_results)

print("## 📄 ASSESSMENT DELIVERABLE REPORT: INTERACTION LOGS ##\n")
for index, row in df.iterrows():
    print(f"### 📍 Task Name: {row['Task']}")
    print(f"**Objective:** {row['Description']}")
    print(f"**Settings Applied:** {row['Settings']}")
    print(f"**Chatbot Output:**\n{row['Bot Response']}")
    print("\n" + "="*60 + "\n")

## 📄 ASSESSMENT DELIVERABLE REPORT: INTERACTION LOGS ##

### 📍 Task Name: 1. Contextual Prompting (No Context)
**Objective:** Basic zero-shot response without background course information.
**Settings Applied:** {'temperature': 0.2}
**Chatbot Output:**
That's a really important question, and the answer can vary quite a


### 📍 Task Name: 2. Contextual Prompting (With Context)
**Objective:** Testing response accuracy when injected with complete course rules.
**Settings Applied:** {'temperature': 0.2}
**Chatbot Output:**
If you miss the final project deadline:

*   There will be an automatic **20% point deduction per day** it is late.
*   You can submit it up to a maximum of **2 days late**.
*   After 2 days late, a grade of **zero** will be assigned for the project.
*   No extensions are granted unless accompanied by a medical exemption.


### 📍 Task Name: 3. Format Constraints (Markdown Structure)
**Objective:** Forcing clean markdown formatting with strict bullet points.
**Settings Ap

In [ ]:
summary_report = """
## 🧠 PROMPT ENGINEERING & LLM BEHAVIOR INSIGHTS REPORT

### 1. The Impact of Contextual Prompting
- **Observation:** Without explicitly set context, the model falls back to general assertions or guesses ("check your syllabus"). With explicit context injected into the system instruction, it quotes specific rules (such as the "20% deduction penalty rule").
- **Insight:** Contextual positioning eliminates guesswork and keeps support bots accurate and grounded.

### 2. Effectiveness of Format Constraints
- **Observation:** Adding structural text constraints forces clean markdown syntax. Setting `response_mime_type="application/json"` safely alters the underlying sampling method to generate verifiable JSON key structures.
- **Insight:** Native application layer configurations (like response_mime_type) are significantly more consistent at formatting data than just writing "respond in JSON" in standard plaintext prompts.

### 3. Iterative Refinement Gains
- **Observation:** Moving from generic system answers to multi-step frameworks containing external links drastically scales utility.
- **Insight:** High-quality system design requires configuring strategic guardrails, links, and step-by-step resolution frameworks.

### 4. Parameter Control Dynamics
- **Observation:** At `temperature=0.2`, the model selects highly common and straightforward explanatory frameworks. At `temperature=0.9`, it chooses unique analogies and stylistic structures.
- **Insight:** Keep temperatures close to 0.0-0.2 for strict functional business routing or policy inquiries, and raise them higher for creative writing, brainstorming assistance, or humanlike conversational variety.
"""

print(summary_report)


## 🧠 PROMPT ENGINEERING & LLM BEHAVIOR INSIGHTS REPORT

### 1. The Impact of Contextual Prompting
- **Observation:** Without explicitly set context, the model falls back to general assertions or guesses ("check your syllabus"). With explicit context injected into the system instruction, it quotes specific rules (such as the "20% deduction penalty rule").
- **Insight:** Contextual positioning eliminates guesswork and keeps support bots accurate and grounded.

### 2. Effectiveness of Format Constraints
- **Observation:** Adding structural text constraints forces clean markdown syntax. Setting `response_mime_type="application/json"` safely alters the underlying sampling method to generate verifiable JSON key structures.
- **Insight:** Native application layer configurations (like response_mime_type) are significantly more consistent at formatting data than just writing "respond in JSON" in standard plaintext prompts.

### 3. Iterative Refinement Gains
- **Observation:** Moving from gener